# Cross-Validation Activity

## Importing Data And Modules

In [ ]:
from palmerpenguins import load_penguins

import statsmodels.api as sm
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score

# Load and prepare the data
penguins = load_penguins()

## Cross-Validation

### CV by Hand

### Setting Up Predictors

In [ ]:
model_data = penguins[
    [
        "species",
        "bill_length_mm",
        "bill_depth_mm",
        "flipper_length_mm",
        "body_mass_g"
    ]
].dropna().reset_index(drop=True)



In [ ]:
# Predictor variables
X = model_data[
    [
        "bill_length_mm",
        "bill_depth_mm"
    ]
]

species_category = pd.Categorical(model_data["species"])

y = pd.Series(
    species_category.codes,
    name="species_code"
)

#### Setting Up CV

In [ ]:
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=123
)


#### Generating the Splits

In [ ]:
splits = list(cv.split(X, y))


#### Examine One Fold

In [ ]:
train_indices, test_indices = splits[0]

#### Creating Subsets

In [ ]:
X_train = X.iloc[train_indices]
X_test = X.iloc[test_indices]

y_train = y.iloc[train_indices]
y_test = y.iloc[test_indices]

#### Add Intercepts

In [ ]:
X_train_constant = sm.add_constant(
    X_train,
    has_constant="add"
)

X_test_constant = sm.add_constant(
    X_test,
    has_constant="add"
)

#### Define Model

In [ ]:
model = sm.MNLogit(
    endog=y_train,
    exog=X_train_constant
)

#### Fit Model

In [ ]:
fitted_model = model.fit(
    method="bfgs",
    maxiter=20000,
    disp=False
)

#### Predict Probabilities

In [ ]:
predicted_probabilities = fitted_model.predict(
    X_test_constant
)

#### Convert Probabilites to Classifications

In [ ]:
predicted_codes = np.asarray(
    predicted_probabilities
).argmax(axis=1)

predicted_codes

#### First Fold Accuracy

In [ ]:
fold_accuracy = accuracy_score(
    y_test,
    predicted_codes
)
fold_accuracy

#### Implementing Loop

In [ ]:
fold_results = []

for fold, (train_indices, test_indices) in enumerate(
    splits,
    start=1
):
    # Select the training and test observations
    X_train = X.iloc[train_indices]
    X_test = X.iloc[test_indices]

    y_train = y.iloc[train_indices]
    y_test = y.iloc[test_indices]

    # Add the intercept
    X_train_constant = sm.add_constant(
        X_train,
        has_constant="add"
    )

    X_test_constant = sm.add_constant(
        X_test,
        has_constant="add"
    )

    # Define and fit the model
    model = sm.MNLogit(
        endog=y_train,
        exog=X_train_constant
    )

    fitted_model = model.fit(
        method="newton",
        maxiter=200,
        disp=False
    )

    # Calculate predicted probabilities
    predicted_probabilities = fitted_model.predict(
        X_test_constant
    )

    # Select the category with the highest probability
    predicted_codes = np.asarray(
        predicted_probabilities
    ).argmax(axis=1)

    # Calculate accuracy
    fold_accuracy = accuracy_score(
        y_test,
        predicted_codes
    )

    # Save the result
    fold_results.append({
        "fold": fold,
        "accuracy": fold_accuracy
    })

#### View Accuracy

In [ ]:
cv_results = pd.DataFrame(fold_results)

cv_results["accuracy"].mean()

### CV via SciKit-Learn

#### Setting Predictors

In [ ]:
model_data = penguins[
    [
        "species",
        "bill_length_mm",
        "bill_depth_mm"
    ]
].dropna()

X = model_data[
    [
        "bill_length_mm",
        "bill_depth_mm"
    ]
]

y = model_data["species"]


#### Setting Model

In [ ]:
multinomial_model = LogisticRegression(
    solver="lbfgs",
    max_iter=20000
)

#### Cross-Validation Set Up

In [ ]:
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=123
)



#### Conducting Cross-Validation

In [ ]:
accuracy_scores = cross_val_score(
    estimator=multinomial_model,
    X=X,
    y=y,
    cv=cv,
    scoring="accuracy"
)